# 04 - Movement Pattern Clustering

**CASEFILE: AI-Powered Missing Person Investigation System**  
*Phase 4: Spatial Stay Point Clustering and Macro-Area Identification*

---

### Overview
In missing person investigations, continuous raw GPS coordinate feeds are noisy and high-dimensional. To extract actionable behavioral patterns, raw coordinates are aggregated into **stay points** (locations where an individual remained stationary within a spatial threshold for a minimum dwell time).

This notebook demonstrates the two-stage clustering pipeline:
1. **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**: Discovers micro-level points of interest and routine dwell clusters using Haversine distance metric without requiring a pre-specified cluster count.
2. **K-Means Clustering**: Groups the resulting micro-cluster centroids into broader **macro-areas** (functional search sectors) evaluated using the **Elbow Method** and **Silhouette Analysis**.
3. **Semantic POI Grounding**: Assigns nearest landmark labels to macro-area centroids to guide ground search operations.

> **DISCLAIMER & ETHICAL STATEMENT**  
> This notebook is part of the CASEFILE academic research framework. All case records, individual profiles, and trajectory logs are synthetic or derived from anonymized public benchmark datasets (GeoLife Beijing). No real-world personally identifiable information (PII), unauthorized surveillance telemetry, or live tracking data is used.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Setup plot styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Working directories
DATA_DIR = '../data' if os.path.exists('../data') else 'data'
REPORTS_DIR = '../reports' if os.path.exists('../reports') else 'reports'
MODELS_DIR = '../models' if os.path.exists('../models') else 'models'

print("Environment initialized successfully.")
print(f"Data Directory: {DATA_DIR}")
print(f"Reports Directory: {REPORTS_DIR}")

## 1. Load Stay Points & Area Centers Data

Stay points represent discrete stops extracted from raw trajectories with parameters:
- Maximum distance threshold: $\approx 200$ meters
- Minimum dwell duration: $\ge 20$ minutes

We load:
- `stay_points_clustered.csv`: Clustered stay points with dwell times, arrival/departure timestamps, and cluster labels.
- `area_centers.csv`: Resulting macro-area centroids labeled with nearest Points of Interest (POIs).

In [ ]:
import sys
sys.path.insert(0, '..')

stay_points_path = os.path.join(DATA_DIR, 'processed', 'stay_points_clustered.csv')
area_centers_path = os.path.join(DATA_DIR, 'processed', 'area_centers.csv')

stay_points_df = pd.read_csv(stay_points_path)
area_centers_df = pd.read_csv(area_centers_path)

print(f"Loaded {len(stay_points_df):,} stay points records.")
print(f"Loaded {len(area_centers_df)} macro-area centroids.")
display(stay_points_df.head(5))

## 2. DBSCAN Clustering Results: 70 Clusters Identified

DBSCAN parameters used:
- $\epsilon = 0.15\text{ km}$ ($150\text{ m}$, converted to radians on earth radius $R = 6371\text{ km}$)
- $\text{MinPts} = 3$ stay points
- Metric: `haversine` distance on spherical coordinates

Points with label `-1` correspond to noise (transient, non-recurrent stops).

In [ ]:
import sys
sys.path.insert(0, '..')

# Segregate valid clusters and noise points
valid_clusters = stay_points_df[stay_points_df['cluster_label'] != -1]
noise_points = stay_points_df[stay_points_df['cluster_label'] == -1]

num_clusters = valid_clusters['cluster_label'].nunique()
num_noise = len(noise_points)
noise_pct = (num_noise / len(stay_points_df)) * 100

print("=" * 45)
print("       DBSCAN CLUSTERING SUMMARY")
print("=" * 45)
print(f"Total stay points:             {len(stay_points_df):,}")
print(f"Number of identified clusters: {num_clusters}")
print(f"Clustered stay points:         {len(valid_clusters):,} ({(len(valid_clusters)/len(stay_points_df))*100:.1f}%)")
print(f"Noise points (label = -1):     {num_noise:,} ({noise_pct:.1f}%)")
print("=" * 45)

# Cluster size distribution
cluster_sizes = valid_clusters['cluster_label'].value_counts().sort_values(ascending=False)
print("\nTop 10 Largest Stay Point Clusters:")
print(cluster_sizes.head(10))

In [ ]:
import sys
sys.path.insert(0, '..')

# Visualize Cluster Size Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Top 15 clusters
top_15 = cluster_sizes.head(15)
ax1.bar(range(len(top_15)), top_15.values, color='royalblue', edgecolor='black', alpha=0.8)
ax1.set_xticks(range(len(top_15)))
ax1.set_xticklabels([f"C{i}" for i in top_15.index], rotation=45)
ax1.set_title("Top 15 Most Frequent DBSCAN Clusters", fontweight='bold')
ax1.set_xlabel("Cluster ID")
ax1.set_ylabel("Number of Stay Points")
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Overall size distribution histogram
ax2.hist(cluster_sizes.values, bins=25, color='teal', edgecolor='black', alpha=0.75)
ax2.set_title("Distribution of All 70 Cluster Sizes", fontweight='bold')
ax2.set_xlabel("Points per Cluster")
ax2.set_ylabel("Number of Clusters")
ax2.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## 3. K-Means Area Clustering: Elbow Method & Silhouette Analysis

While DBSCAN identifies 70 micro-level stay locations, search-and-rescue teams require broader operational sectors. We compute the spatial centroids of all 70 DBSCAN clusters and perform K-Means clustering.

The optimal number of macro-areas $k$ is selected using:
- **Elbow Method (Inertia)**: Finding the point of diminishing return in within-cluster sum-of-squares.
- **Silhouette Score**: Measuring how well separated the area clusters are.

The elbow/silhouette sweep selected **$k = 5$ macro areas** achieving a Silhouette Score of **0.446**.

In [ ]:
import sys
sys.path.insert(0, '..')

# Display the generated Elbow Method and Silhouette report plot
elbow_img = os.path.join(REPORTS_DIR, 'clustering_elbow.png')
if os.path.exists(elbow_img):
    print("Loading reports/clustering_elbow.png:")
    display(Image(filename=elbow_img, width=780))
else:
    print(f"Warning: {elbow_img} not found.")

## 4. Macro Area Centers & POI Grounding

Each of the 5 macro-areas is grounded with nearest Point of Interest (POI) landmarks from Beijing public facility records (parks, transit stations, hospitals) to provide human-interpretable sector labels for search coordinators.

In [ ]:
import sys
sys.path.insert(0, '..')

print("=" * 60)
print("           IDENTIFIED MACRO-AREA SECTORS (K = 5)")
print("=" * 60)
display(area_centers_df[['area_id', 'lat', 'lon', 'name', 'category']])

## 5. Geospatial Visualization: Stay Point Clusters & Area Centers

We plot all stay points in Beijing coordinate space ($39.85^\circ\text{N} - 40.05^\circ\text{N}, 116.20^\circ\text{E} - 116.55^\circ\text{E}$):
- Gray dots: Uncorrelated noise points
- Colored dots: 70 DBSCAN stay point clusters
- Crimson cross markers ($\mathbf{X}$): The 5 K-Means macro-area centroids

In [ ]:
import sys
sys.path.insert(0, '..')

plt.figure(figsize=(12, 9))

# 1. Plot noise points
plt.scatter(noise_points['lon'], noise_points['lat'], 
            c='lightgray', s=12, alpha=0.4, label=f'DBSCAN Noise (n={len(noise_points):,})')

# 2. Plot clustered stay points
scatter = plt.scatter(valid_clusters['lon'], valid_clusters['lat'], 
                      c=valid_clusters['cluster_label'], cmap='tab20', 
                      s=22, alpha=0.75, label=f'70 Stay Point Clusters (n={len(valid_clusters):,})')

# 3. Plot K-Means area centers
plt.scatter(area_centers_df['lon'], area_centers_df['lat'], 
            c='crimson', marker='X', s=200, edgecolors='black', linewidths=1.5,
            label='5 Area Centers (K-Means)', zorder=6)

# Annotate area labels
for _, row in area_centers_df.iterrows():
    plt.annotate(f"Area {int(row['area_id'])}: {row['category']}\n({row['name'].replace('Area near ', '')})", 
                 (row['lon'], row['lat']),
                 textcoords="offset points", xytext=(12, 6),
                 bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="black", lw=0.8, alpha=0.85),
                 fontsize=9, fontweight='bold')

plt.title("Movement Pattern Clustering: 70 DBSCAN Clusters & 5 Macro-Areas", fontsize=13, fontweight='bold')
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(loc='upper left', frameon=True, framealpha=0.9)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 6. Clustering Evaluation Metrics Table

We compute and summarize quantitative clustering validation metrics across both stages.

In [ ]:
import sys
sys.path.insert(0, '..')

# Compute cluster centers of the 70 DBSCAN clusters
cluster_centers = valid_clusters.groupby('cluster_label')[['lat', 'lon']].mean().reset_index()
coords = cluster_centers[['lat', 'lon']].values

# K-Means model with k=5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
k_labels = kmeans.fit_predict(coords)

sil = silhouette_score(coords, k_labels)
ch = calinski_harabasz_score(coords, k_labels)
db = davies_bouldin_score(coords, k_labels)

metrics_summary = pd.DataFrame({
    'Clustering Dimension': [
        'Stage 1: DBSCAN Micro-Clusters',
        'Stage 1: Noise Ratio',
        'Stage 2: K-Means Macro-Areas',
        'K-Means Silhouette Score',
        'Calinski-Harabasz Index',
        'Davies-Bouldin Index'
    ],
    'Value': [
        f"{num_clusters} clusters",
        f"{noise_pct:.2f}%",
        "5 areas",
        f"{sil:.3f}",
        f"{ch:.2f}",
        f"{db:.3f}"
    ],
    'Evaluation / Criterion': [
        'High spatial granularity (eps=150m)',
        'Atypical or transitory one-off stopovers',
        'Optimal k selected via Elbow and Silhouette sweeps',
        'Well-separated clusters (0.446 indicates robust cohesion)',
        'Higher score indicates dense, well-separated clusters',
        'Lower score indicates better cluster partition compactness'
    ]
})

print("=" * 60)
print("           CLUSTERING EVALUATION METRICS TABLE")
print("=" * 60)
display(metrics_summary)

## 7. Investigative Takeaways

- **Dwell Habit Extraction**: DBSCAN condensed 2,102 raw stay points into 70 recurrent functional clusters, filtering out 37% noise representing transitory passing traffic.
- **Search Quadrant Definition**: K-Means consolidated the 70 centers into 5 distinct operational macro-areas ($0, 1, 2, 3, 4$).
- **Pipeline Connection**: These 5 macro-areas serve as the discrete state space for subsequent Markov Chain route modeling (`07_route_prediction.ipynb`) and supervised location prediction (`06_location_prediction.ipynb`).